# OpenPQC — a hands-on tutorial

**Post-quantum cryptography on CPUs and GPUs, from source code to working key exchange and signatures.**

A large quantum computer running Shor's algorithm would break RSA and elliptic-curve cryptography — the
schemes behind today's HTTPS, SSH, and code signing. Data recorded now can be decrypted then
("harvest now, decrypt later"), so the migration has already started. In 2024, NIST standardised three
replacements, and [OpenPQC](https://github.com/UCF-ML-Research/OpenPQC) implements all of them, with a
readable C11/C++20 core and a vendor-neutral SYCL GPU backend:

| Standard | Scheme | Job | Built from | Replaces |
|---|---|---|---|---|
| FIPS 203 | **ML-KEM** (Kyber) | key exchange | module lattices | (EC)DH key agreement |
| FIPS 204 | **ML-DSA** (Dilithium) | signatures | module lattices | RSA / ECDSA signatures |
| FIPS 205 | **SLH-DSA** (SPHINCS+) | signatures | hash functions only | RSA / ECDSA, conservatively |

**In this notebook you will:**

1. Build the library from source (one `cmake` command — already done for you in the Docker image).
2. Call all three schemes interactively from Python: agree on keys, sign messages, and watch tampered
   data get rejected.
3. Compare key and signature sizes against classical cryptography, and measure speed on your machine.
4. Use the C/C++ API directly, and see what the GPU backend changes.

> **Running this notebook.** The smooth path is the Docker image (see
> [`tutorial/README.md`](README.md)): everything below is preinstalled and prebuilt, so every cell runs
> in seconds. Natively you need CMake ≥ 3.20, GCC ≥ 12 or Clang ≥ 16, and
> `pip install jupyterlab matplotlib numpy`. Run the cells top to bottom; each one is safe to re-run.

## 1 · Setup

This cell finds the OpenPQC sources (cloning them if the notebook is running standalone), picks a
C++20 compiler, and checks for CMake. It prints what it found — if it raises, the message says what to
install.

In [ ]:
import os, shutil, subprocess, sys
from pathlib import Path

def run(cmd, tail=40, **kw):
    """Run a command, print the last lines of its output, raise if it fails."""
    cmd = [str(c) for c in cmd]
    print("$", " ".join(cmd))
    r = subprocess.run(cmd, capture_output=True, text=True, **kw)
    out = (r.stdout or "") + (r.stderr or "")
    lines = out.strip().splitlines()
    print("\n".join(lines[-tail:]))
    if r.returncode != 0:
        raise RuntimeError(f"command failed with exit code {r.returncode}")
    return r

# --- find the OpenPQC sources: this notebook normally lives in <repo>/tutorial/
REPO = next((q for q in [Path.cwd(), *Path.cwd().parents]
             if (q / "src/ML/KEM/openpqc_mlkem.h").exists()), None)
if REPO is None:                                   # standalone notebook: fetch the sources
    REPO = Path.cwd() / "OpenPQC"
    if not REPO.exists():
        run(["git", "clone", "--depth", "1",
             "https://github.com/UCF-ML-Research/OpenPQC.git", REPO])
BUILD = REPO / "build-tutorial"

# --- pick a C++20 compiler: GCC >= 12 or Clang >= 16
def major_version(exe):
    try:
        return int(subprocess.run([exe, "-dumpversion"], capture_output=True,
                                  text=True).stdout.strip().split(".")[0])
    except Exception:
        return 0

candidates  = [(os.environ.get("CC", "cc"), os.environ.get("CXX", "c++")), ("gcc", "g++")]
candidates += [(f"gcc-{v}", f"g++-{v}") for v in range(16, 11, -1)]
candidates += [("clang", "clang++")]
candidates += [(f"clang-{v}", f"clang++-{v}") for v in range(21, 15, -1)]
CC = CXX = None
for cc, cxx in candidates:
    if shutil.which(cc) and shutil.which(cxx):
        v = major_version(cxx)
        if ("clang" in cxx and v >= 16) or ("clang" not in cxx and v >= 12):
            CC, CXX = cc, cxx
            break
assert CXX, "No C++20 compiler found: install GCC >= 12 or Clang >= 16 (e.g. apt install g++-13)"
assert shutil.which("cmake"), "CMake not found: install cmake >= 3.20"

print(f"sources   {REPO}")
print(f"compiler  {CC} / {CXX} (major version {major_version(CXX)})")
print(f"cmake     {subprocess.run(['cmake', '--version'], capture_output=True, text=True).stdout.splitlines()[0]}")
print(f"python    {sys.version.split()[0]}")

## 2 · Build the library

One configure and one build. We switch the tests and benchmarks off here so only the library
`openpqc` (built as a shared library, for the Python bridge below) and the example programs are compiled — about half a minute on a laptop, and instant in the
Docker image, where this build is already cached. (The full test suite, which checks OpenPQC byte for
byte against the formally verified `mlkem-native` and the Dilithium and SPHINCS+ reference
implementations, is in section 11.)

In [ ]:
run(["cmake", "-S", REPO, "-B", BUILD,
     f"-DCMAKE_C_COMPILER={CC}", f"-DCMAKE_CXX_COMPILER={CXX}",
     "-DCMAKE_BUILD_TYPE=Release",
     "-DOPENPQC_BUILD_TESTS=OFF", "-DOPENPQC_BUILD_BENCHMARKS=OFF",
     "-DOPENPQC_SYCL=OFF", "-DBUILD_SHARED_LIBS=ON"], tail=6)
run(["cmake", "--build", BUILD, "-j", str(os.cpu_count() or 4)], tail=4)
print("\nbuild complete ✓")

## 3 · A Python bridge for the notebook

OpenPQC is a C/C++ library — in a real application you `#include` its headers and link the library
`openpqc` (section 9 does exactly that). For the interactive parts of this tutorial we also want to
call it from Python: the shared library that section 2 built exports exactly the C API, so we load it
with `ctypes` from the standard library. No bindings to install, nothing to pip-install: the bridge is the
C ABI itself.

In [ ]:
import ctypes
from ctypes import c_char_p, c_int, c_size_t, create_string_buffer

# the shared library of section 2 (BUILD_SHARED_LIBS=ON): it exports the C API and nothing else
SHARED = BUILD / ("libopenpqc.dylib" if sys.platform == "darwin" else "libopenpqc.so")

_lib = ctypes.CDLL(str(SHARED))

def _f(name, *argtypes):
    f = getattr(_lib, name)
    f.argtypes, f.restype = list(argtypes), c_int
    return f

class KEM:
    """ML-KEM (FIPS 203): keygen / encaps / decaps. All functions return 0 (OPENPQC_OK) on success."""
    def __init__(self, name, prefix, pk, sk, ct):
        self.name, self.PK, self.SK, self.CT, self.SS = name, pk, sk, ct, 32
        self._keygen = _f(f"{prefix}_keygen", c_char_p, c_char_p)
        self._keygen_from_seed = _f(f"{prefix}_keygen_from_seed", c_char_p, c_char_p, c_char_p)
        self._encaps = _f(f"{prefix}_encaps", c_char_p, c_char_p, c_char_p)
        self._decaps = _f(f"{prefix}_decaps", c_char_p, c_char_p, c_char_p)
    def keygen(self, coins=None):
        pk, sk = create_string_buffer(self.PK), create_string_buffer(self.SK)
        if coins is None:
            assert self._keygen(pk, sk) == 0
        else:                                       # from a stored seed d || z, 64 bytes
            assert len(coins) == 64 and self._keygen_from_seed(pk, sk, coins) == 0
        return pk.raw, sk.raw
    def encaps(self, pk):
        ct, ss = create_string_buffer(self.CT), create_string_buffer(self.SS)
        assert self._encaps(ct, ss, pk) == 0
        return ct.raw, ss.raw
    def decaps(self, sk, ct):
        ss = create_string_buffer(self.SS)
        assert self._decaps(ss, bytes(ct), sk) == 0
        return ss.raw

class SIG:
    """ML-DSA (FIPS 204) and SLH-DSA (FIPS 205): keygen / sign / verify."""
    def __init__(self, name, prefix, pk, sk, sig):
        self.name, self.PK, self.SK, self.SIG = name, pk, sk, sig
        self._keygen = _f(f"{prefix}_keygen", c_char_p, c_char_p)
        self._sign = _f(f"{prefix}_sign",
                        c_char_p, c_char_p, c_size_t, c_char_p, c_size_t, c_char_p)
        self._verify = _f(f"{prefix}_verify",
                          c_char_p, c_size_t, c_char_p, c_size_t, c_char_p, c_size_t, c_char_p)
    def keygen(self):
        pk, sk = create_string_buffer(self.PK), create_string_buffer(self.SK)
        assert self._keygen(pk, sk) == 0
        return pk.raw, sk.raw
    def sign(self, message, sk, ctx=b""):           # hedged signing, like the standard's default
        sig = create_string_buffer(self.SIG)
        assert self._sign(sig, message, len(message), ctx, len(ctx), sk) == 0
        return sig.raw
    def verify(self, sig, message, pk, ctx=b""):
        return self._verify(bytes(sig), len(sig), bytes(message), len(message),
                            ctx, len(ctx), pk) == 0

MLKEM512   = KEM("ML-KEM-512",  "openpqc_mlkem512",   800, 1632,  768)
MLKEM768   = KEM("ML-KEM-768",  "openpqc_mlkem768",  1184, 2400, 1088)
MLKEM1024  = KEM("ML-KEM-1024", "openpqc_mlkem1024", 1568, 3168, 1568)
MLDSA44    = SIG("ML-DSA-44",   "openpqc_mldsa44",   1312, 2560, 2420)
MLDSA65    = SIG("ML-DSA-65",   "openpqc_mldsa65",   1952, 4032, 3309)
MLDSA87    = SIG("ML-DSA-87",   "openpqc_mldsa87",   2592, 4896, 4627)
SLHDSA128S = SIG("SLH-DSA-SHA2-128s", "openpqc_slhdsa_sha2_128s", 32, 64,  7856)
SLHDSA128F = SIG("SLH-DSA-SHA2-128f", "openpqc_slhdsa_sha2_128f", 32, 64, 17088)

def show(label, data, n=24):
    print(f"{label:<24}{len(data):>7,} bytes   {data[:n].hex()}…")

print("bridge ready ✓  —", SHARED.name, "loaded")

## 4 · ML-KEM (FIPS 203): agreeing on a secret key

A **key encapsulation mechanism** is how two parties who have never met agree on a shared secret over
a public channel — the job Diffie–Hellman does in today's TLS. Browsers already run ML-KEM-768 next to
X25519 on a large share of HTTPS connections.

The flow has three steps, and the secret itself never travels:

1. **Alice** generates a key pair and publishes the public key (her encapsulation key).
2. **Bob** runs `encaps` on it: out come a 32-byte shared secret *and* a ciphertext, which he sends back.
3. **Alice** runs `decaps` on the ciphertext with her secret key — and recovers the same 32 bytes.

Those 32 bytes then key an ordinary symmetric cipher such as AES-256-GCM.

In [ ]:
kem = MLKEM768                                     # the parameter set TLS uses

pk_alice, sk_alice = kem.keygen()                 # 1. Alice, publishing pk
ct, ss_bob         = kem.encaps(pk_alice)          # 2. Bob, sending ct back
ss_alice           = kem.decaps(sk_alice, ct)      # 3. Alice

show("Alice's public key",  pk_alice)
show("Alice's secret key",  sk_alice)
show("ciphertext (Bob→Alice)", ct)
show("Bob's shared secret",   ss_bob)
show("Alice's shared secret", ss_alice)

assert ss_alice == ss_bob
print(f"\nAlice and Bob now share the same {len(ss_bob)}-byte secret ✓")

**What if someone tampers with the ciphertext?** ML-KEM uses *implicit rejection*: `decaps` never
reports an error. Instead it returns a pseudorandom secret unrelated to Bob's, so the attacker gets no
feedback at all — the handshake simply fails later, and no "invalid ciphertext" oracle exists for
chosen-ciphertext attacks.

In [ ]:
tampered = bytearray(ct)
tampered[0] ^= 0x01                                # Eve flips a single bit in transit

ss_after_tamper = kem.decaps(sk_alice, tampered)
show("Bob's shared secret",      ss_bob)
show("secret from tampered ct",  ss_after_tamper)
assert ss_after_tamper != ss_bob
print("\ndecaps 'succeeded' but the secret is unrelated — implicit rejection ✓")

**Reproducibility.** KeyGen also runs from a stored seed (`keygen_from_seed`, the 64 bytes d || z),
and the testing interfaces take every operation's randomness as an argument (`openpqc/testing.h`, for
known-answer tests only). Same coins in, same keys out — this is how the known-answer
tests pin the implementation to the standard, and how the test suite compares OpenPQC byte for byte
with the formally verified `mlkem-native`.

In [ ]:
coins = bytes(range(64))                           # d || z, fixed for the demo
pk1, sk1 = kem.keygen(coins)
pk2, sk2 = kem.keygen(coins)
assert pk1 == pk2 and sk1 == sk2
show("key from fixed coins", pk1)
print("\nsame 64 bytes of randomness → bit-identical keys ✓")

## 5 · ML-DSA (FIPS 204): signing messages

**ML-DSA** is the general-purpose post-quantum signature — the successor to RSA and ECDSA signatures
in certificates, software updates, and documents. A bank signs a payment order; anyone holding the
public key can check it, and changing a single character breaks the signature.

Signing here is *hedged* (the standard's default): each signature mixes in fresh randomness, so
signing the same message twice gives two different, equally valid signatures. The optional **context
string** `ctx` domain-separates signatures, so a signature produced for one application can't be
replayed in another.

In [ ]:
dsa = MLDSA65
pk_bank, sk_bank = dsa.keygen()

order = b"Pay 100 coins to Bob"
sig = dsa.sign(order, sk_bank, ctx=b"payments-v1")

show("bank's public key", pk_bank)
show("signature", sig)
print()
print(f"{'original order verifies:':<44}", dsa.verify(sig, order, pk_bank, ctx=b"payments-v1"))
print(f"{'tampered order (100 → 900 coins):':<44}",
      dsa.verify(sig, b"Pay 900 coins to Bob", pk_bank, ctx=b"payments-v1"))
print(f"{'replayed under another context:':<44}",
      dsa.verify(sig, order, pk_bank, ctx=b"invoices-v2"))

sig2 = dsa.sign(order, sk_bank, ctx=b"payments-v1")
print(f"{'second signature differs (hedged) yet valid:':<44}",
      sig2 != sig and dsa.verify(sig2, order, pk_bank, ctx=b"payments-v1"))

## 6 · SLH-DSA (FIPS 205): signatures from hash functions alone

**SLH-DSA** (SPHINCS+) is the conservative choice: its security rests on nothing but the hash function
(here SHA-256). If lattice cryptanalysis ever surprises us, ML-KEM and ML-DSA fall — SLH-DSA doesn't.
The price is size and speed: a 17 KB signature that takes thousands of hash computations to produce.
That trade-off fits long-lived, high-stakes signatures, like the firmware a device will trust for the
next twenty years.

A vendor signs a 64 KiB firmware image; the device rejects an image with even one flipped bit.

In [ ]:
import os as _os
slh = SLHDSA128F

pk_vendor, sk_vendor = slh.keygen()
firmware = _os.urandom(64 * 1024)                  # stand-in for a 64 KiB firmware image

sig = slh.sign(firmware, sk_vendor, ctx=b"firmware-v2")
show("vendor's public key", pk_vendor)             # tiny keys …
show("signature", sig)                             # … huge signature

backdoored = bytearray(firmware)
backdoored[4096] ^= 0x80                           # one flipped bit, deep inside the image

print()
print(f"{'genuine image verifies:':<32}", slh.verify(sig, firmware, pk_vendor, ctx=b"firmware-v2"))
print(f"{'image with one flipped bit:':<32}", slh.verify(sig, backdoored, pk_vendor, ctx=b"firmware-v2"))

## 7 · What the migration costs: sizes

Post-quantum keys and signatures are bigger than their classical counterparts — that, more than speed,
is the real migration cost (certificate chains, handshake packets, firmware headers). The chart
compares what travels on the wire. Note the log scale: SLH-DSA's signature is ~270× an Ed25519
signature, while its *public key* is as small as a classical one.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

BLUE, ORANGE = "#2a78d6", "#eb6834"                # categorical slots 1 and 2
INK, MUTED, GRID = "#333330", "#6f6e66", "#e5e5e2"

kems = [("X25519  (classical)",   32,   32),
        ("ML-KEM-512",           800,  768),
        ("ML-KEM-768",          1184, 1088),
        ("ML-KEM-1024",         1568, 1568)]
sigs = [("Ed25519  (classical)",  32,    64),
        ("ECDSA P-256  (classical)", 64, 64),
        ("RSA-2048  (classical)", 256,  256),
        ("ML-DSA-44",            1312,  2420),
        ("ML-DSA-65",            1952,  3309),
        ("ML-DSA-87",            2592,  4627),
        ("SLH-DSA-128s",           32,  7856),
        ("SLH-DSA-128f",           32, 17088)]

fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.6),
                         gridspec_kw={"width_ratios": [len(kems), len(sigs)]})
for ax, rows, second, title in [(axes[0], kems, "ciphertext", "Key exchange (KEM)"),
                                (axes[1], sigs, "signature",  "Signatures")]:
    x = np.arange(len(rows))
    pk = [r[1] for r in rows]; snd = [r[2] for r in rows]
    b1 = ax.bar(x - 0.21, pk,  width=0.38, color=BLUE,   label="public key")
    b2 = ax.bar(x + 0.21, snd, width=0.38, color=ORANGE, label=second)
    for bars in (b1, b2):
        for b in bars:
            ax.annotate(f"{b.get_height():,.0f}", (b.get_x() + b.get_width() / 2, b.get_height()),
                        ha="center", va="bottom", fontsize=7.5, color=MUTED)
    ax.set_yscale("log"); ax.set_ylim(20, 60000)
    ax.set_xticks(x, [r[0] for r in rows], rotation=28, ha="right", fontsize=8.5, color=INK)
    ax.set_title(title, color=INK, fontsize=11, loc="left")
    ax.legend(frameon=False, fontsize=8.5, loc="upper left", labelcolor=INK)
    ax.grid(axis="y", color=GRID, linewidth=0.8); ax.set_axisbelow(True)
    ax.tick_params(colors=MUTED, labelsize=8)
    for s in ("top", "right"): ax.spines[s].set_visible(False)
    for s in ("left", "bottom"): ax.spines[s].set_color(GRID)
axes[0].set_ylabel("bytes (log scale)", color=MUTED, fontsize=9)
fig.suptitle("What travels on the wire: post-quantum vs classical", color=INK,
             fontsize=12.5, x=0.065, ha="left")
fig.tight_layout(rect=(0, 0, 1, 0.94))
plt.show()

## 8 · How fast is it? A micro-benchmark

Each operation runs in a loop through the same `ctypes` bridge the demos used (the call overhead is
~1 µs, negligible next to the crypto). Three things to notice:

- **ML-KEM is cheap** — tens of microseconds per operation. Key exchange will not be your bottleneck.
- **ML-DSA signing** is a few times slower than verifying, because signing is a rejection-sampling loop.
- **SLH-DSA signing is ~300× slower than ML-DSA's** and the signature is 5× bigger — the price of
  hash-only security, and exactly the workload the GPU backend attacks in section 10.

Numbers vary a few percent with CPU clocks and load; `bench/run_benchmarks.sh` measures them properly
(cycle counters, medians).

In [ ]:
import time

def ops_per_second(f, budget=0.6):
    """Run f() repeatedly for ~budget seconds, return (ops/s, seconds per op)."""
    f()                                            # warm up
    n, t0 = 0, time.perf_counter()
    while (t := time.perf_counter() - t0) < budget:
        f(); n += 1
    return n / t, t / n

kem_pk, kem_sk = MLKEM768.keygen()
kem_ct, _      = MLKEM768.encaps(kem_pk)
dsa_pk, dsa_sk = MLDSA65.keygen()
dsa_sig        = MLDSA65.sign(b"msg", dsa_sk)
slh_pk, slh_sk = SLHDSA128F.keygen()
slh_sig        = SLHDSA128F.sign(b"msg", slh_sk)

workloads = [
    ("ML-KEM-768 keygen",  lambda: MLKEM768.keygen()),
    ("ML-KEM-768 encaps",  lambda: MLKEM768.encaps(kem_pk)),
    ("ML-KEM-768 decaps",  lambda: MLKEM768.decaps(kem_sk, kem_ct)),
    ("ML-DSA-65 keygen",   lambda: MLDSA65.keygen()),
    ("ML-DSA-65 sign",     lambda: MLDSA65.sign(b"msg", dsa_sk)),
    ("ML-DSA-65 verify",   lambda: MLDSA65.verify(dsa_sig, b"msg", dsa_pk)),
    ("SLH-DSA-128f keygen", lambda: SLHDSA128F.keygen()),
    ("SLH-DSA-128f sign",   lambda: SLHDSA128F.sign(b"msg", slh_sk)),
    ("SLH-DSA-128f verify", lambda: SLHDSA128F.verify(slh_sig, b"msg", slh_pk)),
]

results = []
print(f"{'operation':<24}{'ops/s':>12}{'per op':>14}")
for name, f in workloads:
    ops, per = ops_per_second(f)
    results.append((name, ops))
    print(f"{name:<24}{ops:>12,.0f}{per * 1e6:>11,.0f} µs")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.2))
names = [r[0] for r in results][::-1]
ops   = [r[1] for r in results][::-1]
bars = ax.barh(names, ops, height=0.62, color=BLUE)
for b, v in zip(bars, ops):
    ax.annotate(f" {v:,.0f}", (v, b.get_y() + b.get_height() / 2),
                va="center", fontsize=8.5, color=MUTED)
ax.set_xscale("log"); ax.set_xlim(10, 3e5)
ax.set_title("Operations per second on one CPU core (this machine, log scale)",
             color=INK, fontsize=11.5, loc="left")
ax.grid(axis="x", color=GRID, linewidth=0.8); ax.set_axisbelow(True)
ax.tick_params(colors=MUTED, labelsize=9)
plt.setp(ax.get_yticklabels(), color=INK)
for s in ("top", "right"): ax.spines[s].set_visible(False)
for s in ("left", "bottom"): ax.spines[s].set_color(GRID)
fig.tight_layout()
plt.show()

The repository's own demo prints the same story from C, with per-scheme walkthroughs and one-core
timings — worth running once (about half a minute):

In [ ]:
run([BUILD / "examples" / "demo_cpu"], tail=70)

## 9 · Using the C and C++ API directly

The Python bridge above is a notebook convenience — real users of OpenPQC are C/C++ programs. A
notebook is still a fine place to develop one: write the file with `%%writefile`, then compile and run
it from the next cells. (If you prefer *actual C++ cells*, install a C++ Jupyter kernel such as
[`xeus-cpp`](https://github.com/compiler-research/xeus-cpp) — `conda install -c conda-forge xeus-cpp` —
and include OpenPQC's headers from there. The compile-and-run pattern below needs no extra kernel and
exercises the same toolchain your users will.)

The C++ API puts the sizes into the types, so there are no buffer lengths to get wrong:

In [ ]:
%%writefile pqc_demo.cpp
// The C++ API of OpenPQC: sizes are part of the types, secret keys and shared secrets wipe themselves
// (openpqc::secret<N>), and every call returns an openpqc::status.
#include <cstdio>
#include "openpqc/openpqc.hpp"

int main() {
    using KEM = openpqc::ML_KEM_768;
    KEM::public_key pk; KEM::secret_key sk;
    KEM::ciphertext ct; KEM::shared_secret ss_bob, ss_alice;
    const bool kem_ok = KEM::keygen(pk, sk) == openpqc::status::ok &&
                        KEM::encaps(ct, ss_bob, pk) == openpqc::status::ok &&
                        KEM::decaps(ss_alice, ct, sk) == openpqc::status::ok && ss_bob == ss_alice;
    std::printf("ML-KEM-768: shared secrets %s\n", kem_ok ? "match" : "DIFFER");

    using DSA = openpqc::ML_DSA_65;
    DSA::public_key dpk; DSA::secret_key dsk; DSA::signature sig;
    const std::uint8_t msg[] = "Pay 100 coins to Bob";
    const std::span<const std::uint8_t> m(msg, sizeof msg - 1);
    const bool dsa_ok = DSA::keygen(dpk, dsk) == openpqc::status::ok &&
                        DSA::sign(sig, m, dsk) == openpqc::status::ok &&
                        DSA::verify(sig, m, dpk) == openpqc::status::ok;
    std::printf("ML-DSA-65:  signature %s\n", dsa_ok ? "verifies" : "REJECTED");
    return kem_ok && dsa_ok ? 0 : 1;
}

In [ ]:
run([CXX, "-std=c++20", "-O2", "-I", REPO / "include", "-I", REPO / "src", "-I", BUILD / "include", "pqc_demo.cpp",
     SHARED, f"-Wl,-rpath,{BUILD}", "-o", "pqc_demo"])
run(["./pqc_demo"])

The equivalent C API (`ML/KEM/openpqc_mlkem.h`, `ML/DSA/openpqc_mldsa.h`, `SLH/DSA/openpqc_slhdsa.h`)
is what section 3's bridge calls, one function per algorithm of the standards, commented with the
standards' line numbers — [`examples/demo_cpu.c`](../examples/demo_cpu.c) is the complete reference
program. `openpqc/kem.h` and `openpqc/sig.h` do the same with the parameter set chosen at run time (an
`openpqc_alg`). With CMake, link the target `OpenPQC::openpqc` (`find_package(OpenPQC)` after
`cmake --install`, or `add_subdirectory`).

## 10 · The GPU backend: post-quantum crypto by the thousand

A server terminating TLS or signing certificates doesn't do one operation — it does thousands at a
time, and that's the workload the GPU backend targets. OpenPQC compiles the *same C11 math core* you
just used into **SYCL 2020** kernels, so it runs on NVIDIA and Intel GPUs (and plain CPUs) with either
DPC++ or AdaptiveCpp — an open alternative to NVIDIA's CUDA-only cuPQC. One context type serves all
18 parameter sets (`openpqc/sycl.h`; `openpqc/sycl.hpp` is the C++ layer), and every program follows
the same five steps:

```c
#include "openpqc/sycl.h"                           /* link OpenPQC::sycl */

openpqc_gpu *gpu;
const openpqc_gpu_config cfg = {-1, n};             /* the default device, n operations per launch */
openpqc_gpu_create(&gpu, OPENPQC_ML_DSA_65, &cfg);  /* 1. parameter set, device, batch size */
uint8_t *d_pk = openpqc_gpu_malloc(gpu, n * OPENPQC_MLDSA65_PUBLICKEYBYTES);     /* 2. buffers */
uint8_t *d_sk = openpqc_gpu_malloc(gpu, n * OPENPQC_MLDSA65_SECRETKEYBYTES);
int32_t *d_st = openpqc_gpu_malloc(gpu, n * sizeof(int32_t));                   /* a status per key pair */
openpqc_gpu_sig_keygen(gpu, d_pk, d_sk, d_st, n);   /* 3. ONE call: n key pairs */
openpqc_gpu_memcpy(gpu, pk, d_pk, n * OPENPQC_MLDSA65_PUBLICKEYBYTES);          /* 4. copy back */
openpqc_gpu_free(gpu, d_pk); openpqc_gpu_free(gpu, d_sk); openpqc_gpu_free(gpu, d_st);
openpqc_gpu_destroy(gpu);                           /* 5. done */
```

(Every call returns an `openpqc_status`, and every operation gets its own entry in the status buffer:
a program checks both, as `examples/demo_gpu.cpp` does.)

Measured on a laptop (i7-11370H, RTX 3050 Laptop GPU, DPC++; GPU times include the copies):

| | CPU, 8 threads | GPU, full batch | speed-up |
|---|---:|---:|---:|
| ML-KEM-768 encaps | 74.0 k/s | 945.4 k/s | **12.8×** |
| ML-DSA-65 verify | 25.7 k/s | 440.8 k/s | **17.2×** |
| SLH-DSA-128f keygen | 3.2 k/s | 315.2 k/s | **97.2×** |
| SLH-DSA-128f sign | 140 /s | 10.1 k/s | **72.1×** |

SLH-DSA gains the most: a single signature is thousands of independent hashes, which spread perfectly
over GPU threads. One lone ML-KEM operation, by contrast, is *slower* on the GPU than on a CPU core —
batches are what the GPU is for.

**Building it** needs a SYCL compiler, so it's a native (non-Docker) step — e.g. with DPC++ for an
NVIDIA GPU (`sm_86` is an RTX 30xx; see the top-level README for AdaptiveCpp and more options):

```sh
cmake -S . -B build-gpu -DCMAKE_BUILD_TYPE=Release \
      -DCMAKE_C_COMPILER=<dpcpp>/bin/clang -DCMAKE_CXX_COMPILER=<dpcpp>/bin/clang++ \
      -DOPENPQC_SYCL=ON -DOPENPQC_SYCL_TARGETS=nvptx64-nvidia-cuda,spir64 \
      "-DOPENPQC_SYCL_FLAGS=-Xsycl-target-backend=nvptx64-nvidia-cuda --offload-arch=sm_86" \
      -DOPENPQC_SLHDSA_SYCL_SETS=3
cmake --build build-gpu --target demo_gpu live_demo -j
```

The cell below finds and runs `demo_gpu` if you have built it — it checks every GPU result byte for
byte against the CPU library, then races them. For a talk, `examples/live_demo` tells the same story
with progress bars and a scoreboard (see [`examples/README.md`](../examples/README.md)).

In [ ]:
import glob
RUN_GPU_DEMO = False          # set to True to run it (a few minutes: it validates, then races)

gpu_demos = sorted(glob.glob(str(REPO / "build*" / "examples" / "demo_gpu")))
if not gpu_demos:
    print("No GPU build found (expected build*/examples/demo_gpu).")
    print("Build one with the DPC++ or AdaptiveCpp commands above, then re-run this cell.")
elif RUN_GPU_DEMO:
    run([gpu_demos[0]], tail=80)
else:
    print(f"Found {gpu_demos[0]}")
    print("Set RUN_GPU_DEMO = True in this cell to run it.")

## 11 · Trust, but verify: the test suite

Nobody should take a crypto library's word for it. OpenPQC keeps three upstream implementations
*unmodified* in `reference/` — [mlkem-native](https://github.com/pq-code-package/mlkem-native) (PQCA's
formally verified ML-KEM), reference [Dilithium](https://github.com/pq-crystals/dilithium), and the
designers' [SPHINCS+](https://github.com/sphincs/sphincsplus) — and the tests compare OpenPQC against
them **byte for byte**, plus NIST ACVP and Wycheproof vectors and valgrind-based constant-time checks.
To run them (from the repository root; the first line reconfigures the build with tests on):

```sh
cmake -S . -B build-tutorial -DOPENPQC_BUILD_TESTS=ON
cmake --build build-tutorial -j
ctest --test-dir build-tutorial -LE "network|slow" -j4   # quick offline suite
ctest --test-dir build-tutorial -L network               # ACVP + Wycheproof (downloads vectors)
```

## Where to go next

- **[`examples/`](../examples/README.md)** — `demo_cpu.c`, `demo_gpu.cpp`, and `live_demo` for presentations.
- **Scheme READMEs** — [`src/ML/KEM/`](../src/ML/KEM/README.md),
  [`src/ML/DSA/`](../src/ML/DSA/README.md), [`src/SLH/DSA/`](../src/SLH/DSA/README.md):
  the design of each implementation and its GPU backend, with the experiments behind it.
- **[`bench/`](../bench)** — `run_benchmarks.sh` measures CPU and GPU performance properly on your machine.
- **The standards** — [FIPS 203](https://doi.org/10.6028/NIST.FIPS.203),
  [FIPS 204](https://doi.org/10.6028/NIST.FIPS.204), [FIPS 205](https://doi.org/10.6028/NIST.FIPS.205);
  the code is commented with their algorithm and line numbers, so you can read them side by side.

*OpenPQC is licensed Apache-2.0 OR ISC OR MIT. FN-DSA (Falcon, FIPS 206) is planned once the standard is out.*